In [1]:
from pathlib import Path
import sys
import logging

logger = logging.getLogger(__name__)
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New


In [2]:
import sys 
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

In [3]:
import yaml
import joblib
import pandas as pd

from src.train import (
    train_logistic_regression,
    predict_model,
    tune_logistic_regression,
    train_random_forest,
    calculate_scale_pos_weight,
    train_xgboost,
    save_model
)

from src.evaluate import (
    evaluate_model,
    get_classification_report,
    get_confusion_matrix,
    metrics_to_dataframe,
    confusion_matrix_dataframe
)

In [4]:
CONFIG_PATH = (
    PROJECT_ROOT
    / "config"
    / "config.yaml"
)

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    config = yaml.safe_load(file)

logger.info(
    "Configuration loaded successfully."
)

2026-09-21 18:55:20,599 - INFO - Configuration loaded successfully.


In [5]:
features_path = (
    PROJECT_ROOT
    / config["paths"]["features"]
)

logging.info(
    "Features directory:%s",
    features_path
)

2026-09-21 18:55:20,754 - INFO - Features directory:c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\models\features


In [6]:
X_train_processed = joblib.load(
    features_path
    / "X_train_processed.joblib"
)

X_val_processed = joblib.load(
    features_path
    / "X_val_processed.joblib"
)

X_test_processed = joblib.load(
    features_path
    / "X_test_processed.joblib"
)

y_train = joblib.load(
    features_path
    / "y_train.joblib"
)

y_val = joblib.load(
    features_path
    / "y_val.joblib"
)

y_test = joblib.load(
    features_path
    / "y_test.joblib"
)

feature_names = joblib.load(
    features_path
    / "feature_names.joblib"
)

preprocessor = joblib.load(
    features_path
    / "preprocessor.joblib"
)

In [7]:
logger.info("X_train:%s", X_train_processed.shape)
logger.info("X_val:%s", X_val_processed.shape)
logger.info("X_test:%s", X_test_processed.shape)

logger.info("------------------------")

logger.info("y_train:%s", y_train.shape)
logger.info("y_val:%s", y_val.shape)
logger.info("y_test:%s", y_test.shape)

logger.info("------------------------")

logger.info(
    "Number of features:%s",
    len(feature_names)
)

2026-09-21 18:55:22,106 - INFO - X_train:(67533, 3702)
2026-09-21 18:55:22,106 - INFO - X_val:(14471, 3702)
2026-09-21 18:55:22,106 - INFO - X_test:(14472, 3702)
2026-09-21 18:55:22,121 - INFO - ------------------------
2026-09-21 18:55:22,121 - INFO - y_train:(67533,)
2026-09-21 18:55:22,133 - INFO - y_val:(14471,)
2026-09-21 18:55:22,133 - INFO - y_test:(14472,)
2026-09-21 18:55:22,133 - INFO - ------------------------
2026-09-21 18:55:22,148 - INFO - Number of features:3702


In [8]:
random_state = config[
    "project"
]["random_state"]

max_iter = config[
    "training"
]["model"]["max_iter"]

baseline_C = config[
    "training"
]["baseline"]["C"]

baseline_class_weight = config[
    "training"
]["baseline"]["class_weight"]

logger.info("Random state:%s", random_state)
logger.info("Max iterations:%s", max_iter)
logger.info("Baseline C:%s", baseline_C)
logger.info(
    "Baseline class weight:%s",
    baseline_class_weight
)

2026-09-21 18:55:22,426 - INFO - Random state:42
2026-09-21 18:55:22,426 - INFO - Max iterations:1000
2026-09-21 18:55:22,426 - INFO - Baseline C:1.0
2026-09-21 18:55:22,437 - INFO - Baseline class weight:balanced


In [9]:
baseline_model = train_logistic_regression(
    X_train=X_train_processed,
    y_train=y_train,
    C=baseline_C,
    class_weight=baseline_class_weight,
    max_iter=max_iter,
    random_state=random_state
)

logger.info(
    "Baseline model trained successfully."
)

2026-09-21 18:55:42,799 - INFO - Baseline model trained successfully.


In [10]:
(
    baseline_val_pred,
    baseline_val_proba,
    baseline_metrics
) = evaluate_model(
    baseline_model,
    X_val_processed,
    y_val
)

In [11]:
logger.info("Baseline Validation Results")
logger.info("=" * 40)

for metric, value in baseline_metrics.items():
    logger.info(
        f"{metric}: {value:.4f}"
    )

2026-09-21 18:55:43,065 - INFO - Baseline Validation Results
2026-09-21 18:55:43,079 - INFO - ========================================
2026-09-21 18:55:43,079 - INFO - accuracy: 0.6564
2026-09-21 18:55:43,079 - INFO - precision: 0.1229
2026-09-21 18:55:43,094 - INFO - recall: 0.5273
2026-09-21 18:55:43,105 - INFO - f1: 0.1994
2026-09-21 18:55:43,109 - INFO - roc_auc: 0.6263
2026-09-21 18:55:43,110 - INFO - pr_auc: 0.1322


In [12]:
logger.info(
    get_classification_report(
        y_val,
        baseline_val_pred
    )
)

2026-09-21 18:55:43,476 - INFO -               precision    recall  f1-score   support

     On-time       0.94      0.67      0.78     13297
        Late       0.12      0.53      0.20      1174

    accuracy                           0.66     14471
   macro avg       0.53      0.60      0.49     14471
weighted avg       0.87      0.66      0.73     14471



In [13]:
baseline_cm = get_confusion_matrix(
    y_val,
    baseline_val_pred
)

logger.info("Confusion Matrix:")
logger.info(baseline_cm)

2026-09-21 18:55:43,587 - INFO - Confusion Matrix:
2026-09-21 18:55:43,590 - INFO - [[8880 4417]
 [ 555  619]]


In [14]:
baseline_results = metrics_to_dataframe(
    metrics=baseline_metrics,
    model_name="Logistic Regression - Baseline",
    C=baseline_C,
    class_weight=baseline_class_weight
)


In [15]:
C_values = config[
    "training"
]["tuning"]["C_values"]

class_weights = config[
    "training"
]["tuning"]["class_weights"]

logger.info("C values:%s", C_values)
logger.info("Class weights:%s", class_weights)

2026-09-21 18:55:44,262 - INFO - C values:[0.01, 0.1, 1.0, 10.0]
2026-09-21 18:55:44,262 - INFO - Class weights:['balanced', None]


In [16]:
(
    tuning_results_df,
    best_model,
    best_C,
    best_class_weight
) = tune_logistic_regression(
    X_train=X_train_processed,
    y_train=y_train,
    X_val=X_val_processed,
    y_val=y_val,
    C_values=C_values,
    class_weights=class_weights,
    max_iter=max_iter,
    random_state=random_state
)

c:\Users\Anwar Altorkmani\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Anwar Altorkmani\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\linear_model\_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https:/

In [17]:
logger.info(
    "Number of configurations:%s",
    len(tuning_results_df)
)

logger.info(
    tuning_results_df[
        [
            "C",
            "class_weight",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc"
        ]
    ]
)

2026-09-21 18:57:41,271 - INFO - Number of configurations:8
2026-09-21 18:57:41,271 - INFO -        C class_weight  precision    recall        f1   roc_auc    pr_auc
0   0.10     balanced   0.126965  0.536627  0.205346  0.635591  0.137973
1   0.01     balanced   0.125482  0.526405  0.202656  0.630814  0.137808
2   1.00     balanced   0.122915  0.527257  0.199356  0.626313  0.132170
3  10.00     balanced   0.119705  0.539182  0.195915  0.618115  0.125212
4  10.00         None   0.162791  0.005963  0.011504  0.622948  0.126406
5   0.01         None   0.000000  0.000000  0.000000  0.619252  0.133075
6   0.10         None   0.000000  0.000000  0.000000  0.634523  0.138015
7   1.00         None   0.000000  0.000000  0.000000  0.633236  0.134870


In [18]:
logger.info("Best configuration:")
logger.info(
    "C:%s",
    best_C
)

logger.info(
    "Class weight:%s",
    best_class_weight
)

logger.info(
    "Selection metric:%s",
    config["training"]["selection_metric"]
)

2026-09-21 18:57:41,541 - INFO - Best configuration:
2026-09-21 18:57:41,552 - INFO - C:0.1
2026-09-21 18:57:41,552 - INFO - Class weight:balanced
2026-09-21 18:57:41,567 - INFO - Selection metric:f1


In [19]:
(
    y_val_pred,
    y_val_proba,
    best_val_metrics
) = evaluate_model(
    best_model,
    X_val_processed,
    y_val
)

In [20]:
logger.info("Validation Results")
logger.info("------------------")

for metric, value in best_val_metrics.items():
    logger.info(
        f"{metric:<10}: {value:.4f}"
    )

2026-09-21 18:57:42,114 - INFO - Validation Results
2026-09-21 18:57:42,120 - INFO - ------------------
2026-09-21 18:57:42,129 - INFO - accuracy  : 0.6631
2026-09-21 18:57:42,136 - INFO - precision : 0.1270
2026-09-21 18:57:42,136 - INFO - recall    : 0.5366
2026-09-21 18:57:42,136 - INFO - f1        : 0.2053
2026-09-21 18:57:42,151 - INFO - roc_auc   : 0.6356
2026-09-21 18:57:42,151 - INFO - pr_auc    : 0.1380


In [21]:
from src.evaluate import get_classification_report


logger.info(
    get_classification_report(
        y_val,
        y_val_pred
    )
)

2026-09-21 18:57:42,528 - INFO -               precision    recall  f1-score   support

     On-time       0.94      0.67      0.79     13297
        Late       0.13      0.54      0.21      1174

    accuracy                           0.66     14471
   macro avg       0.53      0.61      0.50     14471
weighted avg       0.88      0.66      0.74     14471



In [22]:
cm = get_confusion_matrix(
    y_val,
    y_val_pred
)

logger.info("Confusion Matrix:")
logger.info(cm)

2026-09-21 18:57:42,653 - INFO - Confusion Matrix:
2026-09-21 18:57:42,653 - INFO - [[8965 4332]
 [ 544  630]]


In [23]:
validation_results = metrics_to_dataframe(
    metrics=best_val_metrics,
    model_name="Logistic Regression",
    C=best_C,
    class_weight=best_class_weight
)

validation_results

,model,C,class_weight,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,0.1,balanced,0.66305,0.126965,0.536627,0.205346,0.635591,0.137973


In [24]:
validation_results.to_csv(
    features_path
    / "validation_results.csv",
    index=False
)

tuning_results_df.to_csv(
    features_path
    / "tuning_results.csv",
    index=False
)

In [25]:
random_state = config[
    "project"
]["random_state"]

rf_n_estimators = config[
    "random_forest"
]["n_estimators"]

rf_max_depth = config[
    "random_forest"
]["max_depth"]

rf_min_samples_split = config[
    "random_forest"
]["min_samples_split"]

rf_min_samples_leaf = config[
    "random_forest"
]["min_samples_leaf"]
rf_class_weight = config[
    "random_forest"
]["class_weight"]
rf_n_jobs = config[
    "random_forest"
]["n_jobs"]

logger.info("Random state:%s", random_state)
logger.info("Max iterations:%s", rf_n_estimators)
logger.info("Max Depth:%s",rf_max_depth)
logger.info("Min Samples Splits:%s",rf_min_samples_split)
logger.info("Min Samples Leaf:%s",rf_min_samples_leaf)
logger.info("Class Weight:%s",rf_class_weight)
logger.info("N Jobs:%s",rf_n_jobs)

2026-09-21 18:58:01,081 - INFO - Random state:42
2026-09-21 18:58:01,084 - INFO - Max iterations:300
2026-09-21 18:58:01,091 - INFO - Max Depth:None
2026-09-21 18:58:01,093 - INFO - Min Samples Splits:2
2026-09-21 18:58:01,097 - INFO - Min Samples Leaf:1
2026-09-21 18:58:01,100 - INFO - Class Weight:balanced
2026-09-21 18:58:01,103 - INFO - N Jobs:-1


In [27]:
rf_model = train_random_forest(X_train=X_train_processed,
          y_train=y_train,
          n_estimators=rf_n_estimators,
          random_state=random_state,
          )

In [28]:
rf_val_pred,rf_val_proba,rf_metrics = evaluate_model(rf_model,X_val_processed,y_val)

In [ ]:
logger.info("Random Forest Validation Results")
logger.info("=" * 40)

for metric, value in rf_metrics.items():
    logger.info(
        f"{metric}: {value:.4f}"
    )



2026-09-21 19:14:31,714 - INFO - Baseline Validation Results
2026-09-21 19:14:33,754 - INFO - ========================================
2026-09-21 19:14:33,755 - INFO - accuracy: 0.9187
2026-09-21 19:14:33,755 - INFO - precision: 0.4483
2026-09-21 19:14:33,755 - INFO - recall: 0.0111
2026-09-21 19:14:33,769 - INFO - f1: 0.0216
2026-09-21 19:14:33,771 - INFO - roc_auc: 0.6329
2026-09-21 19:14:33,772 - INFO - pr_auc: 0.1426


In [30]:
logger.info(
    get_classification_report(
        y_val,
        rf_val_pred
    )
)

2026-09-21 19:14:56,729 - INFO -               precision    recall  f1-score   support

     On-time       0.92      1.00      0.96     13297
        Late       0.45      0.01      0.02      1174

    accuracy                           0.92     14471
   macro avg       0.68      0.50      0.49     14471
weighted avg       0.88      0.92      0.88     14471



In [32]:
baseline_cm = get_confusion_matrix(
    y_val,
    rf_val_pred
)

logger.info("Confusion Matrix:")
logger.info(baseline_cm)

2026-09-21 19:15:14,763 - INFO - Confusion Matrix:
2026-09-21 19:15:14,766 - INFO - [[13281    16]
 [ 1161    13]]


In [35]:
random_state = config[
    "project"
]["random_state"]

xgb_estimators = config[
    "xgboost"
]["n_estimators"]

xgb_max_depth = config[
    "xgboost"
]["max_depth"]

xgb_learning_rate = config[
    "xgboost"
]["learning_rate"]

xgb_subsample = config[
    "xgboost"
]["subsample"]
xgb_colsample_bytree = config[
    "xgboost"
]["colsample_bytree"]

xgb_scale_pos_weight = config[
    "xgboost"
]["scale_pos_weight"]

xgb_objective = config[
    "xgboost"
]["objective"]

xgb_eval_metric = config[
    "xgboost"
]["eval_metric"]

xgb_n_jobs = config[
    "xgboost"
]["n_jobs"]

logger.info("Random state:%s", random_state)
logger.info("Max iterations:%s", xgb_estimators)
logger.info("Max Depth:%s",xgb_max_depth)
logger.info("Learning Rate:%s",xgb_learning_rate)
logger.info("Sub Sample:%s",xgb_subsample)
logger.info("Colsample_bytree:%s",xgb_colsample_bytree)
logger.info("Scale_pos_weight:%s",xgb_scale_pos_weight)
logger.info("Objective:%s",xgb_objective)
logger.info("Eval_Metric:%s",xgb_eval_metric)
logger.info("N Jobs:%s",xgb_n_jobs)

2026-09-21 19:18:58,914 - INFO - Random state:42


2026-09-21 19:18:58,939 - INFO - Max iterations:300
2026-09-21 19:18:59,039 - INFO - Max Depth:6
2026-09-21 19:18:59,055 - INFO - Learning Rate:0.05
2026-09-21 19:18:59,154 - INFO - Sub Sample:0.8
2026-09-21 19:18:59,158 - INFO - Colsample_bytree:0.8
2026-09-21 19:18:59,162 - INFO - Scale_pos_weight:11.325789377623654
2026-09-21 19:18:59,166 - INFO - Objective:binary:logistic
2026-09-21 19:18:59,171 - INFO - Eval_Metric:log_loss
2026-09-21 19:18:59,175 - INFO - N Jobs:-1


In [36]:
xgb_model = train_xgboost(X_train=X_train_processed,
          y_train=y_train,
          n_estimators=xgb_estimators,
          random_state=random_state,
          max_depth=xgb_max_depth,
          learning_rate=xgb_learning_rate)

In [37]:
(
    xgb_val_pred,
    xgb_val_proba,
    xgb_metrics
) = evaluate_model(
    xgb_model,
    X_val_processed,
    y_val
)

In [ ]:
logger.info("XGBoost Validation Results")
logger.info("=" * 40)

for metric, value in xgb_metrics.items():
    logger.info(
        f"{metric}: {value:.4f}"
    )

2026-09-21 19:21:20,482 - INFO -  XGBoost Validation Results
2026-09-21 19:21:20,486 - INFO - ========================================
2026-09-21 19:21:20,489 - INFO - accuracy: 0.6948
2026-09-21 19:21:20,491 - INFO - precision: 0.1356
2026-09-21 19:21:20,493 - INFO - recall: 0.5136
2026-09-21 19:21:20,495 - INFO - f1: 0.2145
2026-09-21 19:21:20,497 - INFO - roc_auc: 0.6509
2026-09-21 19:21:20,499 - INFO - pr_auc: 0.1442


In [39]:
logger.info(
    get_classification_report(
        y_val,
        xgb_val_pred
    )
)

2026-09-21 19:22:14,728 - INFO -               precision    recall  f1-score   support

     On-time       0.94      0.71      0.81     13297
        Late       0.14      0.51      0.21      1174

    accuracy                           0.69     14471
   macro avg       0.54      0.61      0.51     14471
weighted avg       0.88      0.69      0.76     14471



In [40]:
baseline_cm = get_confusion_matrix(
    y_val,
    xgb_val_pred
)

logger.info("Confusion Matrix:")
logger.info(baseline_cm)

2026-09-21 19:22:35,999 - INFO - Confusion Matrix:
2026-09-21 19:22:36,002 - INFO - [[9452 3845]
 [ 571  603]]


In [42]:
joblib.dump(
    xgb_model,
    features_path
    / "best_model_xgboost.joblib"
)
joblib.dump(
    best_model,
    features_path
    / "best_model_linear_regression_after_tuning.joblib"
)
joblib.dump(
    rf_model,
    features_path
    / "rf_model.joblib"
)
logger.info(
    "Model and validation artifacts saved successfully."
)

2026-09-21 19:29:50,202 - INFO - Model and validation artifacts saved successfully.


In [53]:
(
    y_test_pred,
    y_test_proba,
    test_metrics_xgb
) = evaluate_model(
    xgb_model,
    X_test_processed,
    y_test
)

In [54]:
logger.info("Final Test Results")
logger.info("------------------")

for metric, value in test_metrics_xgb.items():
    logger.info(
        f"{metric:<10}: {value:.4f}"
    )

2026-09-21 19:37:46,082 - INFO - Final Test Results
2026-09-21 19:37:46,087 - INFO - ------------------
2026-09-21 19:37:46,091 - INFO - accuracy  : 0.7011
2026-09-21 19:37:46,095 - INFO - precision : 0.1389
2026-09-21 19:37:46,099 - INFO - recall    : 0.5162
2026-09-21 19:37:46,105 - INFO - f1        : 0.2189
2026-09-21 19:37:46,108 - INFO - roc_auc   : 0.6651
2026-09-21 19:37:46,116 - INFO - pr_auc    : 0.1412


In [55]:
logger.info(
    get_classification_report(
        y_test,
        y_test_pred
    )
)

2026-09-21 19:37:55,531 - INFO -               precision    recall  f1-score   support

     On-time       0.94      0.72      0.82     13298
        Late       0.14      0.52      0.22      1174

    accuracy                           0.70     14472
   macro avg       0.54      0.62      0.52     14472
weighted avg       0.88      0.70      0.77     14472



In [56]:
cm_test = get_confusion_matrix(
    y_test,
    y_test_pred
)

logger.info("Test Confusion Matrix:")
logger.info(cm_test)

2026-09-21 19:38:01,374 - INFO - Test Confusion Matrix:
2026-09-21 19:38:01,432 - INFO - [[9540 3758]
 [ 568  606]]


In [57]:
final_results = metrics_to_dataframe(
    metrics=test_metrics_xgb,
    model_name="XGBoost",
)

final_results

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,XGBoost,0.701078,0.138863,0.516184,0.218852,0.665091,0.141238


In [58]:
final_results.to_csv(
    features_path
    / "final_test_results.csv",
    index=False
)

logger.info(
    "Final test results saved successfully."
)

2026-09-21 19:39:07,950 - INFO - Final test results saved successfully.


In [59]:
cm_df = confusion_matrix_dataframe(
    cm_test
)

cm_df.to_csv(
    features_path
    / "test_confusion_matrix.csv"
)

logger.info(
    "Confusion matrix saved successfully."
)

2026-09-21 19:39:09,787 - INFO - Confusion matrix saved successfully.


In [60]:
report = get_classification_report(
    y_test,
    y_test_pred
)

report_dict = (
    __import__("sklearn.metrics", fromlist=["classification_report"])
)


In [61]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_test_pred,
    target_names=[
        "On-time",
        "Late"
    ],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report
).transpose()

report_df.to_csv(
    features_path
    / "test_classification_report.csv"
)

logger.info(
    "Classification report saved successfully."
)

2026-09-21 19:39:21,951 - INFO - Classification report saved successfully.


In [62]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_test_pred,
    target_names=[
        "On-time",
        "Late"
    ],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report
).transpose()

report_df.to_csv(
    features_path
    / "test_classification_report.csv"
)

logger.info(
    "Classification report saved successfully."
)

2026-09-21 19:39:26,442 - INFO - Classification report saved successfully.
